<a href="https://colab.research.google.com/github/iraisheredia/IA-APLICACA-A-MODELOS/blob/main/Copia_de_Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


Cliente de Groq inicializado correctamente.


# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq 'q'



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.6 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


Cliente de Groq inicializado correctamente.


### **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "Cual es la diferencia entre la RAM y el almacenamiento en una computadora?"
print(prompt)

Cual es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b", # Updated model as requested. Please check Groq's deprecation page for current models if this fails: https://console.groq.com/docs/deprecations
    messages=[{"role": "user", "content": prompt}]
)

In [ ]:
print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio)**  
- **Volatilidad:** Se borra cuando la computadora se apaga.  
- **Velocidad:** Es mucho más rápida (gigahertz) que cualquier forma de almacenamiento permanente.  
- **Capacidad:** Generalmente está en el rango de GBs (4 GB, 8 GB, 16 GB, etc.) y suele ser más cara por gigabyte que el almacenamiento.  
- **Uso:** Almacena temporalmente el sistema operativo, las aplicaciones que están ejecutándose y los datos con los que estas aplicaciones están trabajando en ese momento.  
- **Acceso:** El procesador lee y escribe datos en la RAM a través del bus de datos; es la “memoria de trabajo” de la CPU.

**Almacenamiento (disco duro HDD, unidad de estado sólido SSD, NVMe, etc.)**  
- **Volatilidad:** Conserva los datos aunque la energía se corte.  
- **Velocidad:** Mucho más lenta que la RAM (megabytes por segundo), aunque los SSD modernos ya alcanzan decenas de gigabytes por segundo, muy por debajo de la RAM.  
- **Capacidad:** Puede llegar a cientos de gig

In [ ]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-18e5b2e5-de6d-4cf0-80ca-7e3d71d1fa60",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**RAM (Memoria de Acceso Aleatorio)**  \n- **Volatilidad:** Se borra cuando la computadora se apaga.  \n- **Velocidad:** Es mucho más rápida (gigahertz) que cualquier forma de almacenamiento permanente.  \n- **Capacidad:** Generalmente está en el rango de GBs (4 GB, 8 GB, 16 GB, etc.) y suele ser más cara por gigabyte que el almacenamiento.  \n- **Uso:** Almacena temporalmente el sistema operativo, las aplicaciones que están ejecutándose y los datos con los que estas aplicaciones están trabajando en ese momento.  \n- **Acceso:** El procesador lee y escribe datos en la RAM a través del bus de datos; es la “memoria de trabajo” de la CPU.\n\n**Almacenamiento (disco duro HDD, unidad de estado sólido SSD, NVMe, etc.)**  \n- **Volatilidad:** Conserva los datos aunque la energía se corte.  \n- **Velocida

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:",response.usage.prompt_tokens)
print("Tokens de la respuesta:",response.usage.completion_tokens)
print("Total de tokens:",response.usage.total_tokens)

Tokens del prompt: 86
Tokens de la respuesta: 760
Total de tokens: 846


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
model="openai/gpt-oss-20b", # Modelo actualizado. Si falla, por favor revisa la página de deprecaciones de Groq: https://console.groq.com/docs/deprecations
messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
print(f"Tiempo de respuesta:{duracion: .2f} segundos")

Tiempo de respuesta: 1.26 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
model="openai/gpt-oss-20b", # Modelo actualizado. Si falla, por favor revisa la página de deprecaciones de Groq: https://console.groq.com/docs/deprecations
messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.26 s — 846 tokens
Modelo grande: 1.39 s — 1133 tokens

Respuesta del modelo grande:
 **RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco duro, SSD, etc.)**

| Característica | RAM | Almacenamiento |
|-----------------|-----|----------------|
| **Propósito** | Guarda los datos y programas que la CPU necesita usar de inmediato. | Guarda de forma permanente los datos, archivos y sistemas operativos. |
| **Volatilidad** | **Volátil**: pierde todo el contenido cuando el ordenador se apaga o reinicia. | **No volátil**: conserva la información incluso sin energía. |
| **Velocidad** | Extremadamente rápida (nanosegundos). | Más lenta: SSD (milisegundos) > HDD (decenas de milisegundos). |
| **Capacidad** | Generalmente limitada (4 GB, 8 GB, 16 GB, etc.) | Muy mayor: 256 GB, 512 GB, 1 TB, 2 TB, etc. |
| **Costo por GB** | Más caro por gigabyte. | Menor costo por gigabyte. |
| **Tecnología** | DRAM (Dynamic RAM), SRAM en cachés, etc. | NAND flash en SSD, platos magnétic

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets

# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq 'q'



In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [ ]:
# Definir la lista de preguntas
preguntas = [
    "Cual es la diferencia entre estudiar ingeneria de software y ciencia de datos?",
    "Cual es el resultado de sumar 25 mas 32?",
    "Cual es la diferencia entre un perro raza Pug y un perro raza Alaska"
]
for p in preguntas:
    print(p)

Cual es la diferencia entre estudiar ingeneria de software y ciencia de datos?
Cual es el resultado de sumar 25 mas 32?
Cual es la diferencia entre un perro raza Pug y un perro raza Alaska


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
import time

current_prompt = preguntas[0]

inicio_q1 = time.time()
response_q1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": current_prompt}]
)
duracion_q1 = time.time() - inicio_q1

resultado_1 = {
    "pregunta": current_prompt,
    "respuesta": response_q1.choices[0].message.content,
    "tiempo_respuesta": duracion_q1,
    "prompt_tokens": response_q1.usage.prompt_tokens,
    "completion_tokens": response_q1.usage.completion_tokens,
    "total_tokens": response_q1.usage.total_tokens
}

print(f"Pregunta 1: {resultado_1['pregunta']}")
print(f"Tiempo de respuesta: {resultado_1['tiempo_respuesta']:.2f} segundos")
print(f"Tokens del prompt: {resultado_1['prompt_tokens']}")
print(f"Tokens de la respuesta: {resultado_1['completion_tokens']}")
print(f"Total de tokens: {resultado_1['total_tokens']}")

Pregunta 1: Cual es la diferencia entre estudiar ingeneria de software y ciencia de datos?
Tiempo de respuesta: 2.60 segundos
Tokens del prompt: 88
Tokens de la respuesta: 2048
Total de tokens: 2136


In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print("Tokens del prompt:",response.usage.prompt_tokens)
print("Tokens de la respuesta:",response.usage.completion_tokens)
print("Total de tokens:",response.usage.total_tokens)

Tokens del prompt: 87
Tokens de la respuesta: 1688
Total de tokens: 1775


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
import time

current_prompt = preguntas[1]

inicio_q2 = time.time()
response_q2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": current_prompt}]
)
duracion_q2 = time.time() - inicio_q2

resultado_2 = {
    "pregunta": current_prompt,
    "respuesta": response_q2.choices[0].message.content,
    "tiempo_respuesta": duracion_q2,
    "prompt_tokens": response_q2.usage.prompt_tokens,
    "completion_tokens": response_q2.usage.completion_tokens,
    "total_tokens": response_q2.usage.total_tokens
}

print(f"Pregunta 2: {resultado_2['pregunta']}")
print(f"Tiempo de respuesta: {resultado_2['tiempo_respuesta']:.2f} segundos")
print(f"Tokens del prompt: {resultado_2['prompt_tokens']}")
print(f"Tokens de la respuesta: {resultado_2['completion_tokens']}")
print(f"Total de tokens: {resultado_2['total_tokens']}")

Pregunta 2: Cual es el resultado de sumar 25 mas 32?
Tiempo de respuesta: 0.48 segundos
Tokens del prompt: 84
Tokens de la respuesta: 46
Total de tokens: 130


In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
import time

current_prompt = preguntas[2]

inicio_q3 = time.time()
response_q3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": current_prompt}]
)
duracion_q3 = time.time() - inicio_q3

resultado_3 = {
    "pregunta": current_prompt,
    "respuesta": response_q3.choices[0].message.content,
    "tiempo_respuesta": duracion_q3,
    "prompt_tokens": response_q3.usage.prompt_tokens,
    "completion_tokens": response_q3.usage.completion_tokens,
    "total_tokens": response_q3.usage.total_tokens
}

print(f"Pregunta 3: {resultado_3['pregunta']}")
print(f"Tiempo de respuesta: {resultado_3['tiempo_respuesta']:.2f} segundos")
print(f"Tokens del prompt: {resultado_3['prompt_tokens']}")
print(f"Tokens de la respuesta: {resultado_3['completion_tokens']}")
print(f"Total de tokens: {resultado_3['total_tokens']}")

Pregunta 3: Cual es la diferencia entre un perro raza Pug y un perro raza Alaska
Tiempo de respuesta: 2.24 segundos
Tokens del prompt: 87
Tokens de la respuesta: 1654
Total de tokens: 1741


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

Para aclarar la respuesta de la **Pregunta 1** del reto, aquí tienes su contenido:

In [ ]:
print(f"Respuesta a la Pregunta 1: {resultado_1['respuesta']}")

Respuesta a la Pregunta 1: ### Diferencias básicas

| Aspecto | Ingeniería de Software | Ciencia de Datos |
|---------|------------------------|------------------|
| **Enfoque principal** | **Construir, mantener y escalar software** (aplicaciones, sistemas, infraestructuras). | **Extraer conocimiento e insights** a partir de datos, construir modelos predictivos y prescriptivos. |
| **Objetivo del proyecto** | Entregar un producto funcional, fiable y fácil de mantener. | Obtener valor accionable o tomar decisiones basadas en datos. |
| **Herramientas típicas** | Git, CI/CD, Docker, Kubernetes, JIRA, IDEs (IntelliJ, VS Code), lenguajes como Java, C#, Python, Go. | Jupyter, Pandas, NumPy, scikit‑learn, TensorFlow/PyTorch, R, SQL, Spark, Tableau/Power BI. |
| **Metodologías** | Scrum/Agile, Kanban, TDD, BDD, diseño de arquitectura (MVC, microservicios). | Experimentos controlados, validación cruzada, análisis estadístico, storytelling con datos. |
| **Perfil del profesional** | **Arquitect

In [ ]:
print(f"Respuesta a la Pregunta 2: {resultado_2['respuesta']}")

Respuesta a la Pregunta 2: La suma de 25 más 32 es **57**.


In [ ]:
print(f"Respuesta a la Pregunta 3: {resultado_3['respuesta']}")

Respuesta a la Pregunta 3: ### Pug vs. Alaskan (Alaskan Malamute / Alaskan Husky)

| Característica | Pug | Alaskan Malamute / Alaskan Husky |
|-----------------|-----|----------------------------------|
| **Tamaño** | Pequeño‑mediano (30‑35 lb / 13‑16 kg, 12‑14 in / 30‑36 cm de altura) | Grande (50‑80 lb / 23‑36 kg, 21‑25 in / 53‑64 cm de altura) |
| **Coat (pelaje)** | Corto, liso, de una sola capa; se mantiene suave y no necesita mucho mantenimiento. | Doble capa: capa externa áspera y capa interna densa. Requiere cepillado regular, especialmente en épocas de muda. |
| **Temperamento** | Cariñoso, sociable, leal, a veces “travesura” y “técnico” (¡te ama a ti, pero no a la comida!). Muy adaptable a la vida en apartamento. | Independiente, fuerte, a veces “cazar” y “trabajar”. Necesita mucho ejercicio y estimulación mental. Puede ser dominante con otros perros si no se socializa bien. |
| **Salud** | Problemas respiratorios (braquicefalia), problemas oculares (ojo rojo, lagrimeo), dis

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print("Resultados de todas las preguntas:")
for i, res in enumerate(resultados):
    print(f"\n--- Pregunta {i+1} ---")
    print(f"Pregunta: {res['pregunta']}")
    print(f"Tiempo de respuesta: {res['tiempo_respuesta']:.2f} segundos")
    print(f"Tokens del prompt: {res['prompt_tokens']}")
    print(f"Tokens de la respuesta: {res['completion_tokens']}")
    print(f"Total de tokens: {res['total_tokens']}")
    print(f"Respuesta del modelo: {res['respuesta']}")

Resultados de todas las preguntas:

--- Pregunta 1 ---
Pregunta: Cual es la diferencia entre estudiar ingeneria de software y ciencia de datos?
Tiempo de respuesta: 2.60 segundos
Tokens del prompt: 88
Tokens de la respuesta: 2048
Total de tokens: 2136
Respuesta del modelo: ### Diferencias básicas

| Aspecto | Ingeniería de Software | Ciencia de Datos |
|---------|------------------------|------------------|
| **Enfoque principal** | **Construir, mantener y escalar software** (aplicaciones, sistemas, infraestructuras). | **Extraer conocimiento e insights** a partir de datos, construir modelos predictivos y prescriptivos. |
| **Objetivo del proyecto** | Entregar un producto funcional, fiable y fácil de mantener. | Obtener valor accionable o tomar decisiones basadas en datos. |
| **Herramientas típicas** | Git, CI/CD, Docker, Kubernetes, JIRA, IDEs (IntelliJ, VS Code), lenguajes como Java, C#, Python, Go. | Jupyter, Pandas, NumPy, scikit‑learn, TensorFlow/PyTorch, R, SQL, Spark, Tableau/P

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados y concluir
import pandas as pd

# Crear un DataFrame para una mejor visualización
df_resultados = pd.DataFrame(resultados)

print("\n--- Tabla Resumen de Resultados ---")
print(df_resultados[['pregunta', 'tiempo_respuesta', 'total_tokens']])

print("\n--- Conclusión ---")

conclusion_texto = ""

# Evaluación de la Pregunta 1 (Diferencia Software vs Ciencia de Datos)
if "ingeniería de software" in resultado_1['respuesta'].lower() and "ciencia de datos" in resultado_1['respuesta'].lower():
    conclusion_texto += "La Pregunta 1 (diferencias entre Ingeniería de Software y Ciencia de Datos) fue respondida de manera satisfactoria, ofreciendo una explicación detallada de ambas disciplinas.\n"
else:
    conclusion_texto += "La Pregunta 1 podría necesitar revisión; no se encontraron los términos clave en la respuesta.\n"

# Evaluación de la Pregunta 2 (Suma 25 + 32)
if "57" in resultado_2['respuesta']:
    conclusion_texto += "La Pregunta 2 (suma de 25 más 32) fue resuelta **correctamente** con el resultado esperado (57).\n"
else:
    conclusion_texto += "La Pregunta 2 no dio el resultado correcto (se esperaba 57). Puede que el modelo no sea óptimo para cálculos directos.\n"

# Evaluación de la Pregunta 3 (Diferencia Pug vs Alaska)
if "pug" in resultado_3['respuesta'].lower() and ("alaskan malamute" in resultado_3['respuesta'].lower() or "alaskan husky" in resultado_3['respuesta'].lower()):
    conclusion_texto += "La Pregunta 3 (diferencias entre Pug y raza 'Alaska') fue bien abordada, incluso señalando la ambigüedad de 'Alaska' y comparando con razas comunes.\n"
else:
    conclusion_texto += "La Pregunta 3 podría necesitar revisión; no se encontraron los términos clave de comparación de razas.\n"

conclusion_texto += "\nEn general, el modelo `openai/gpt-oss-20b` demuestra ser competente para preguntas de conocimiento general y comparaciones, aunque su rendimiento en cálculos directos debe ser verificado con más pruebas. Los tiempos de respuesta fueron aceptables y los tokens consumidos varían según la complejidad de la respuesta."

print(conclusion_texto)


--- Tabla Resumen de Resultados ---
                                            pregunta  tiempo_respuesta  \
0  Cual es la diferencia entre estudiar ingeneria...          1.871017   
1           Cual es el resultado de sumar 25 mas 32?          0.484781   
2  Cual es la diferencia entre un perro raza Pug ...          2.240147   

   total_tokens  
0          1444  
1           130  
2          1741  

--- Conclusión ---
La Pregunta 1 (diferencias entre Ingeniería de Software y Ciencia de Datos) fue respondida de manera satisfactoria, ofreciendo una explicación detallada de ambas disciplinas.
La Pregunta 2 (suma de 25 más 32) fue resuelta **correctamente** con el resultado esperado (57).
La Pregunta 3 (diferencias entre Pug y raza 'Alaska') fue bien abordada, incluso señalando la ambigüedad de 'Alaska' y comparando con razas comunes.

En general, el modelo `openai/gpt-oss-20b` demuestra ser competente para preguntas de conocimiento general y comparaciones, aunque su rendimiento en cál